# Benchmark embedders para semantic similarity search

In [ ]:
#Inic
all_docs = [] # <--- Lista para acumular todos los documentos

for filename in os.listdir(SOURCE_DOCS_PATH): # <--- Iteramos sobre los archivos
    if filename.endswith(".md"): # <--- Filtramos solo archivos .md
        file_path = os.path.join(SOURCE_DOCS_PATH, filename) # <--- Construimos la ruta completa
        processed_document_chunks = process_markdown_document(file_path) # <--- Procesamos cada archivo
        all_docs.extend(processed_document_chunks) # <--- Agregamos los documentos
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000, 
    chunk_overlap=100,
    separators=["\n\n"])

chunked_splits = text_splitter.split_documents(all_docs)

# Add metadata
for i, doc in enumerate(chunked_splits):
    doc.metadata["doc_id"] = f"chunk_{i}"

In [ ]:
#Codificacion de los chunks y Creacion de la base de datos
if len(os.listdir(PERSIST_DIRECTORY))==0:
    vectorStore = EmbeddDocsAndPersist(chunked_splits,embedding_encoder,PERSIST_DIRECTORY)
else:
    vectorStore = load_persisted_db(embedding_encoder,PERSIST_DIRECTORY)

In [ ]:
GGUF_MODEL_PATH = "/home/nico/.cache/llama.cpp/nomic-embed-text-v2-moe.Q6_K.gguf"
PERSIST_DIRECTORY = "./db_chroma/nomic_moe"

embedding_encoder = LlamaCppEmbeddings(
    model_path=GGUF_MODEL_PATH,
    n_gpu_layers=0,   # Offload layers to GPU on your Windows machine. Set to 0 for CPU.
    n_batch=64,       # The number of documents to process in a single batch.
    n_ctx=512,        # The context size of the model.
    verbose=False      # Set to True to see more detailed llama.cpp output.
)
vectorStore = EmbeddDocsAndPersist(chunked_splits,embedding_encoder,PERSIST_DIRECTORY)

In [ ]:
max_tokens = 150
temperature = 0
top_p = 0.05
echo = False
stop = ["\n\n"]

evaluation_dataset_AIgenerated=[]

for chunk in  chunked_splits:
    chunkContent=chunk.page_content
    chunkId=chunk.metadata.get('doc_id')
    user_prompt= f"Redacta una pregunta simple sobre el contenido del siguiente texto. La pregunta se utilizara para evaluar la retrieval de un vectorstore. Solo genera la pregunta.  Texto: {chunkContent}"
    messages = [HumanMessage(content=user_prompt)]
    model_output = llm.invoke(messages)    
    final_result = model_output.content.strip()
    print(f"pedido: {user_prompt} \n --- \n respuesta: {final_result}")
    evaluation_dataset_AIgenerated.append({"question":f"{final_result}","ground_truth_doc_id":f"{chunkId}"})

In [ ]:
def evaluate_vectorstore_as_retriever(eval_dataset, vector_store, k=5):
    """
    Evaluates the performance of a retriever using a given dataset.

    Args:
        eval_dataset (list): A list of dictionaries with "question" and "ground_truth_doc_id".
        vector_store: The ChromaDB vector store instance.
        k (int): The number of top documents to retrieve for evaluation.

    Returns:
        dict: A dictionary containing the calculated metrics.
    """
    hits = 0
    reciprocal_ranks = []
    misses = [] # To store information about failed queries for later analysis

    print(f"Starting evaluation for k={k}...")

    for item in eval_dataset:
        question = item["question"]
        ground_truth_id = item["ground_truth_doc_id"]
        
        # Perform the similarity search
        # The result is a list of tuples: [(Document, score), (Document, score), ...]
        retrieved_docs_with_scores = vector_store.similarity_search_with_score(question, k=k)
        
        # Extract the doc_ids from the metadata of the retrieved documents
        retrieved_ids = [doc.metadata.get('doc_id') for doc, score in retrieved_docs_with_scores]
        
        # Check if the ground truth ID is in the retrieved IDs
        if ground_truth_id in retrieved_ids:
            hits += 1
            # Find the rank (position) of the correct document. Ranks are 1-based.
            rank = retrieved_ids.index(ground_truth_id) + 1
            reciprocal_ranks.append(1 / rank)
        else:
            reciprocal_ranks.append(0)
            misses.append({
                "question": question,
                "expected": ground_truth_id,
                "retrieved": retrieved_ids
            })

    total_questions = len(eval_dataset)
    hit_rate = (hits / total_questions) * 100
    mrr = np.mean(reciprocal_ranks)

    return {
        "hit_rate_at_k": k,
        "hit_rate": f"{hit_rate:.2f}%",
        "mrr": f"{mrr:.4f}",
        "total_questions": total_questions,
        "hits": hits,
        "misses_count": len(misses),
        "misses": misses
    }

In [ ]:
evaluate_vectorstore_as_retriever(evaluation_dataset_AIgenerated, vectorStore, k=5)